# 📋 Notebook 6 — Final Evaluation & Model Comparison

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from transformers import BertForSequenceClassification, BertTokenizer
from sklearn.metrics import confusion_matrix, classification_report
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# Load saved BERT model
model_path = '../saved_models/bert_model'
model     = BertForSequenceClassification.from_pretrained(model_path).to(device)
tokenizer = BertTokenizer.from_pretrained(model_path)
model.eval()
print('✅ Model loaded!')

In [ ]:
# Predict function
label_names = {0:'World',1:'Sports',2:'Business',3:'Technology'}

def predict(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    pred = logits.argmax(dim=1).item()
    prob = torch.softmax(logits, dim=1)[0][pred].item()
    return label_names[pred], round(prob, 4)

In [ ]:
# Test with new texts
test_cases = [
    'Tesla announces record quarterly profits driven by strong EV demand',
    'Manchester United wins the Champions League in a stunning comeback',
    'WHO declares new health emergency over rising respiratory infections',
    'Google unveils Gemini Ultra 2.0 at annual developer conference'
]
print('\n🔍 Predictions:')
for text in test_cases:
    cat, conf = predict(text)
    print(f'  [{cat} | {conf:.0%}] {text[:60]}')

In [ ]:
# Model comparison chart
models  = ['TF-IDF + LogReg', 'TextCNN', 'BiLSTM', 'BERT (fine-tuned)']
scores  = [0.845, 0.890, 0.882, 0.934]
colors  = ['#aec7e8','#ffbb78','#98df8a','#4C72B0']

plt.figure(figsize=(9,5))
bars = plt.bar(models, scores, color=colors, edgecolor='black', width=0.5)
for bar, score in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{score:.1%}', ha='center', fontweight='bold')
plt.ylim(0.75, 1.0)
plt.title('Model Accuracy Comparison — AG News', fontsize=14)
plt.ylabel('Accuracy')
plt.tight_layout()
os.makedirs('../outputs/plots', exist_ok=True)
plt.savefig('../outputs/plots/model_comparison.png', dpi=150)
plt.show()
print('✅ Comparison chart saved!')